In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()
final_path = BASE_DIR / "Replication Project" / "Data" / "Final" / "acs_ssc_final_v2.pkl"

df = pd.read_pickle(final_path)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print(df.head())

Shape: (38279, 98)
Columns: 98
    year  serial  statefip  countyfip   hhwt  sscouple_mar  sscouple_coh  \
0   2012     442         1          0   65.0         False          True   
2   2012     836         1          0  100.0         False          True   
4   2012    1162         1          0   19.0         False          True   
6   2012    1671         1          0  163.0         False          True   
10  2012    2396         1         97   61.0          True         False   

    sscouple_all  sex  age  ...  migplac1  hinsemp  staterecog  staterecog2  \
0           True    1   41  ...         0        1           0            0   
2           True    2   44  ...         0        2           0            0   
4           True    2   37  ...         0        2           0            0   
6           True    1   53  ...         0        1           0            0   
10          True    1   56  ...         0        2           0            0   

    staterecog_policy  staterecog_tim

In [3]:
df.duplicated(["year", "serial"]).sum()

np.int64(0)

In [5]:
df["sscouple_mar"].mean()


np.float64(0.42182397659290993)

In [6]:
df["sscouple_mar"].sum()


np.int64(16147)

In [7]:
df["sscouple_coh"].sum()

np.int64(22132)

In [8]:
(df["c_agemin"] < 18).sum(), (df["c_agemax"] > 60).sum()

(np.int64(0), np.int64(0))

In [9]:
(df["preW"] + df["postWpreO"] + df["postO"]).value_counts()

1    38279
Name: count, dtype: int64

In [10]:
(
    df["c_children0_1"] + df["c_children1_5"] + df["c_children6_18"]
    == df["c_children"]
).mean()

np.float64(0.9998432560934194)

In [11]:
print("Duplicate couples:", df.duplicated(["year", "serial"]).sum())
print("Married share:", df["sscouple_mar"].mean())
print("Married count:", df["sscouple_mar"].sum())
print("Cohabiting count:", df["sscouple_coh"].sum())
print("Age min violations:", (df["c_agemin"] < 18).sum())
print("Age max violations:", (df["c_agemax"] > 60).sum())
print("Period check:")
print((df["preW"] + df["postWpreO"] + df["postO"]).value_counts())

Duplicate couples: 0
Married share: 0.42182397659290993
Married count: 16147
Cohabiting count: 22132
Age min violations: 0
Age max violations: 0
Period check:
1    38279
Name: count, dtype: int64


In [13]:
def federal_tax(income, filing_status, n_children):
    # standard deduction
    if filing_status == "single":
        deduction = 12000
    else:  # joint
        deduction = 24000

    taxable = max(income - deduction, 0)

    # simple progressive tax
    if taxable <= 20000:
        tax = 0.1 * taxable
    elif taxable <= 60000:
        tax = 0.1 * 20000 + 0.15 * (taxable - 20000)
    else:
        tax = 0.1 * 20000 + 0.15 * 40000 + 0.25 * (taxable - 60000)

    # child tax credit
    credit = 2000 * n_children
    tax = max(tax - credit, 0)

    return tax

In [14]:
def state_tax(income, recognized):
    if recognized == 0:
        return 0
    return 0.05 * income

In [15]:
def marriage_subsidy(row):
    y1 = row["r_incearn"]
    y2 = row["sp_incearn"]
    children = row["c_children"]

    # assign children to primary earner when single
    if y1 >= y2:
        c1, c2 = children, 0
    else:
        c1, c2 = 0, children

    # SINGLE taxes
    t1 = federal_tax(y1, "single", c1)
    t2 = federal_tax(y2, "single", c2)

    st1 = state_tax(y1, row["staterecog_policy"])
    st2 = state_tax(y2, row["staterecog_policy"])

    single_total = t1 + t2 + st1 + st2

    # JOINT taxes
    joint_income = y1 + y2

    t_joint = federal_tax(joint_income, "joint", children)
    st_joint = state_tax(joint_income, row["staterecog_policy"])

    joint_total = t_joint + st_joint

    return single_total - joint_total

In [16]:
df["marriage_subsidy"] = df.apply(marriage_subsidy, axis=1)

In [17]:
print(df["marriage_subsidy"].describe())
print((df["marriage_subsidy"] > 0).mean())  # subsidy share
print((df["marriage_subsidy"] < 0).mean())  # penalty share

count    38279.000000
mean     -1601.282550
std       2929.201823
min      -7000.000000
25%      -3800.000000
50%       -650.000000
75%        210.000000
max       5125.000000
Name: marriage_subsidy, dtype: float64
0.27087959455576166
0.6058413229185715


In [19]:
features = [
    "age", "sp_age",
    "r_yrsedu", "sp_yrsedu",
    "r_race", "sp_race",
    "r_occbroad", "sp_occbroad",
    "r_degbroad", "sp_degbroad",
    "c_children",
    "c_agediff", "c_edudiff",
    "statefip", "year"
]

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
X = pd.get_dummies(df[features], columns=["statefip", "year"], drop_first=True)
y = df["r_incearn"]

# optional but recommended
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [23]:
lasso = LassoCV(cv=5, n_jobs=-1)
lasso.fit(X_scaled, y)

df["income_hat"] = lasso.predict(X_scaled)

In [25]:
import numpy as np
print(df["income_hat"].describe())
print(np.corrcoef(df["r_incearn"], df["income_hat"])[0,1])

count     38279.000000
mean      65141.868100
std       31911.383147
min     -111537.366580
25%       42425.067178
50%       67152.014400
75%       89375.729992
max      151743.810955
Name: income_hat, dtype: float64
0.3998134986906232
